# Проект: Рынок видеоигр

- Автор: Деньгина Анна Дмитриевна
- Дата: 26.08.2026

### Цели и задачи проекта

**Основная цель проекта:** обработка и первичный анализ данных рынка игровой индустрии с 2000 до 2013 года для дальнейшего исследования. Для этого необходимо изучить структуру данных, провести их предобработку и фильтрацию для получения необходимого среза, а также категоризировать данные.

### Описание данных

Данные `/datasets/new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:

- `Name` — название игры.

- `Platform` — название платформы.

- `Year of Release` — год выпуска игры.

- `Genre` — жанр игры.

- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).

- `EU sales` — продажи в Европе (в миллионах проданных копий).

- `JP sales` — продажи в Японии (в миллионах проданных копий).

- `Other sales` — продажи в других странах (в миллионах проданных копий).

- `Critic Score` — оценка критиков (от 0 до 100).

- `User Score` — оценка пользователей (от 0 до 10).

- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

[1. Загрузка данных и знакомство с ними](#loading)

[2. Проверка ошибок в данных и их предобработка](#preprocessing)

[3. Фильтрация данных](#filtration)

[4. Категоризация данных](#categorization)

[5. Итоговый вывод](#conclusion)

---

<a id="loading"></a>
## 1. Загрузка данных и знакомство с ними

- Загружаем необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
# Выгружаем new_games.csv в датафрейм
df = pd.read_csv('/datasets/new_games.csv')

- Выведем первые строки и результат метода `info()`.


In [3]:
print(df.head())

                       Name Platform  Year of Release         Genre  NA sales  \
0                Wii Sports      Wii           2006.0        Sports     41.36   
1         Super Mario Bros.      NES           1985.0      Platform     29.08   
2            Mario Kart Wii      Wii           2008.0        Racing     15.68   
3         Wii Sports Resort      Wii           2009.0        Sports     15.61   
4  Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing     11.27   

  EU sales JP sales  Other sales  Critic Score User Score Rating  
0    28.96     3.77         8.45          76.0          8      E  
1     3.58     6.81         0.77           NaN        NaN    NaN  
2    12.76     3.79         3.29          82.0        8.3      E  
3    10.93     3.28         2.95          80.0          8      E  
4     8.89    10.22         1.00           NaN        NaN    NaN  


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Датасет `/datasets/new_games.csv` содержит 11 столбцов и 16596 строк, в которых представлена информация о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки.

Изучим типы данных и их корректность:

- **Строковые данные (object).** 7 столбцов имеют тип данных `object`: 

    - `Name`, `Platform`, `Genre`, `Rating` содержат строковую информацию о названии игры, названии платформы и рейтинге в организации ESRB. Тип данных выбран верно.
    
    - `EU sales` и `JP sales` содержат данные о количестве продаж в миллионах. Такие данные рекомендуется привести к типу `float64` для удобства расчетов.
    
    - `User Score` содержит данные об оценках пользователей, его также следует привести к типу `float64` для удобства расчетов.


- **Числовые значения с плавающей запятой (float64).** 4 столбца имеют тип данных `float64`:
    
    - `Year of Release` содержит информацию о годе выпуска игры. Тип данных `float64` не подходит, так как год не может быть дробным, следует привести к целочисленному типу с пониженной разрядностью `int32`.
    - `NA sales` и `Other sales` содержат информацию о количестве продаж в миллионах. Тип данных выбран верно.
    - `Critic Score` содержит информацию об оценке критиков от 0 до 100. Тип данных выбран верно.

Предварительный анализ данных показал, что пропуски встречаются в 6 столбцах: `Name`, `Year of Release`, `Genre`, `Critic Score`, `User Score`, `Rating`. 

Названия столбцов отражают содержимое столбцов и не требуют переименования. Однако, требуется привести их к единому стилю `sneaker case`, т.е. к нижнему регистру и заменить пробелы на `_`.

---
<a id="preprocessing"></a>

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

- Выводим на экран названия всех столбцов датафрейма

In [5]:
# Выведем названия столбцов
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

- Приводим все столбцы к стилю `snake case`. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [6]:
# Приводим названия столбцов к стилю sneaker case
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
# Выведем результат метода info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  object 
 6   jp_sales         16956 non-null  object 
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  object 
 10  rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


### 2.2. Типы данных


- `year_of_release` содержит информацию о годе выпуска игры, который имеет тип данных `float64` из-за пропусков или некорректности внесения данных, например "2001.0", либо "2001.3".

In [7]:
# Выведем список уникальных значений года выпуска
df['year_of_release'].unique()

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

In [8]:
# Рассчитаем количество пропусков в абсолютном значении
df['year_of_release'].isna().sum()

275

In [9]:
# Рассчитаем долю пропусков в процентах, окргулим до 2 знаков после запятой
round(df['year_of_release'].isna().sum() / len(df) * 100, 2)

1.62

Данные внесены корректно, в значениях столбца `year_of_release` нет дробных чисел. Соответственно перед преобразованием типа данных к целочисленному необходимо провести обработку пропусков. 

Так как в дальнейшем анализ данных будет производится по срезу с 2000 до 2013 года включительно, а доля пропусков в столбце составляет 1.62%, то было принято решение удалить строки с пропущенными значениями. Их удаление не исказит данные и даст возможность преобразовать год в целые числа.

Также следует сохранить первоначальное количество строк в переменной для дальнейших расчетов.

In [10]:
# Создадим переменную total_row_cnt с расчетом общего количества строк датафрейма до предобработки данных
total_row_cnt = len(df)

# Удаляем строки с пропущенным значением года выпуска
df = df.dropna(subset=['year_of_release'])
# Преобразуем столбец year_of_release к целочисленному типу  с минимальной размерностью
df['year_of_release'] = pd.to_numeric(df['year_of_release'], downcast = 'integer')

In [11]:
# Выведем список уникальных значений года выпуска
df['year_of_release'].unique()

array([2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010,
       2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014,
       1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000, 1995, 1991,
       1981, 1987, 1980, 1983], dtype=int16)

- В числовых столбцах `eu_sales`, `jp_sales` и `user_score` встречаются строковые значения, например `unknown` или другие. Приведем их к числовому типу данных, заменив строковые значения на пропуски.

In [12]:
# Приведение к числовому типу столбца eu_sales
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
# Приведение к числовому типу столбца jp_sales
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
# Приведение к числовому типу столбца user_score
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

После преобразования некорректных типов данных, выведем информацию о датафрейме на экран

In [13]:
# Выведем результат метода info()
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16681 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16681 non-null  object 
 2   year_of_release  16681 non-null  int16  
 3   genre            16679 non-null  object 
 4   na_sales         16681 non-null  float64
 5   eu_sales         16675 non-null  float64
 6   jp_sales         16677 non-null  float64
 7   other_sales      16681 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float64
 10  rating           9901 non-null   object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.4+ MB


### 2.3. Наличие пропусков в данных

- Рассчитаем количество пропущенных строк в абсолютном значении


In [14]:
missing_percentage = round(df.isna().sum() / len(df) * 100,2) 
missing_df = pd.DataFrame({
    'Количество пропусков': df.isna().sum(),
    'Процент пропусков (%)': missing_percentage
})
print("\nАнализ пропущенных значений:")
display(missing_df)


Анализ пропущенных значений:


,Количество пропусков,Процент пропусков (%)
name,2,0.01
platform,0,0.00
year_of_release,0,0.00
genre,2,0.01
na_sales,0,0.00
eu_sales,6,0.04
jp_sales,4,0.02
other_sales,0,0.00
critic_score,8596,51.53
user_score,9123,54.69


#### Промежуточный вывод

Пропуски встречаются в 7 столбцах: `name`, `genre`, `eu_sales`, `jp_sales`, `critic_score`, `user_score`, `rating`. После преобразования типов данных в столбце `year_of_release` не осталось пропусков, а в столбцах `eu_sales`, `jp_sales`, `user_score` они появились. 

Наибольшее количество пропусков наблюдается в столбцах `critic_score` - 51.5%, `user_score` - 54.7%, `rating` - 40.6%. Это может быть связано с тем, что не все игры получили оценки от пользователей и критиков, либо не имеют возрастного рейтинга ESPB. Столбцы с таким количеством пропущенных значений стоит оставить без изменения, либо заменить на значения-ндикаторы, но в дальнейшем учесть это при категоризации данных.

В столбцах `name` и `genre` 2 пропущенных значения (около 0.012%), при анализе данных названия игры и жанра являются ключевыми параметрами, поэтому их можно удалить.

В столбцах `eu_sales` и `jp_sales` 6 (0.036%) и 4 (0.024%) пропусков соответственно. Мы можем заменить их на среднее значение в зависимости от платформы и года выпуска игры.

#### Обработка пропусков

1. Удаляем строки с пропущенными значения названия игры и жанра.

In [16]:
# Удаляем строки с пропущенными значениями в столбце name
df = df.dropna(subset=['name'])
# Удаляем строки с пропущенными значениями в столбце genre
df = df.dropna(subset=['genre'])

2. Заменяем пропущенные значения в столбцах `eu_sales` и `jp_sales` на среднее значение в зависимости от платформы и года выпуска игры.

In [17]:
# Считаем средние продажи по платформе и году для EU и JP
eu_mean = df.groupby(['platform', 'year_of_release'])['eu_sales'].mean()
jp_mean = df.groupby(['platform', 'year_of_release'])['jp_sales'].mean()

In [18]:
# Создаем функцию для заполнения пропусков в столбце eu_sales
def fill_eu(row):
    if pd.isna(row['eu_sales']):
        return eu_mean.loc[(row['platform'], row['year_of_release'])]
    else:
        return row['eu_sales']
# Создаем функцию для заполнения пропусков в столбце jp_sales
def fill_jp(row):
    if pd.isna(row['jp_sales']):
        return jp_mean.loc[(row['platform'], row['year_of_release'])]
    else:
        return row['jp_sales']

In [19]:
# Применяем функции для заполнения пропусков
df['eu_sales'] = df.apply(fill_eu, axis = 1)
df['jp_sales'] = df.apply(fill_jp, axis = 1)

3. Заменяем пропущенные значения в столбцах `critic_score`и `user_score` на значение- индикатор `-1`. 

In [20]:
# Заполняем столбец critic_score значением-индикатором -1
df['critic_score'] = df['critic_score'].fillna(-1)
# Заполняем столбец user_score значением-индикатором -1
df['user_score'] = df['user_score'].fillna(-1)

In [21]:
# Выводим количество пропущенных строк в датафрейме
df.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score          0
user_score            0
rating             6778
dtype: int64

После проведения обработки пропущенных значений в датафрейме, пропуски значений остались только в столбце `rating`, так как он не используется в расчётах, а замена значений нецелесообразна. 

### 2.4. Явные и неявные дубликаты в данных

**Проверка наличия неявных дубликатов**

- Изучим уникальные значения в категориальных данных: названия жанра игры, платформы, рейтинга и года выпуска. Проверим, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.

In [22]:
# Выведем список уникальных значений жанра
print(df['genre'].unique())
print('Количество уникальных значений - ', len(df['genre'].unique()))

['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']
Количество уникальных значений -  24


In [23]:
# Выведем список уникальных значений названия платформы
print(df['platform'].unique())
print('Количество уникальных значений - ', len(df['platform'].unique()))

['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
Количество уникальных значений -  31


In [24]:
# Выведем список уникальных значений рейтинга ESPB
print(df['rating'].unique())
print('Количество уникальных значений - ', len(df['rating'].unique()))

['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
Количество уникальных значений -  9


In [25]:
# Выведем список уникальных значений года выпуска
print(df['year_of_release'].unique())
print('Количество уникальных значений - ', len(df['year_of_release'].unique()))

[2006 1985 2008 2009 1996 1989 1984 2005 1999 2007 2010 2013 2004 1990
 1988 2002 2001 2011 1998 2015 2012 2014 1992 1997 1993 1994 1982 2016
 2003 1986 2000 1995 1991 1981 1987 1980 1983]
Количество уникальных значений -  37


Нормализуем данные с текстовыми значениями в столбцах `genre` и `platform`:

In [26]:
# Приведем названия жанра игры к нижнему регистру
df['platform'] = df['platform'].str.upper()
# Выведем список уникальных значений после преобразования
print(df['platform'].unique())
print('Количество уникальных значений - ', len(df['platform'].unique()))

['WII' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XONE' 'WIIU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
Количество уникальных значений -  31


In [27]:
# Приведем названия платформы к верхнему регистру 
df['genre'] = df['genre'].str.lower()
# Выведем список уникальных значений после преобразования
print(df['genre'].unique())
print('Количество уникальных значений - ', len(df['genre'].unique()))

['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']
Количество уникальных значений -  12


Неявные дубликаты были обнаружены только в столбце `genre`, после приведения значений к нижнему регистру осталось 12 уникальных значений (в 2 раза меньше). Также в столбце `platform` значения были приведены к верхнему регистру для корректной группировки.

**Проверка наличия явных дубликатов в данных**

In [28]:
# Считаем количество явных дубликатов в датафрейме
print(f"Количество явных дубликатов в датафрейме - {df.duplicated().sum()}")

Количество явных дубликатов в датафрейме - 235


**Промежуточный вывод:** в датафрейме обнаружено 235 явных дубликатов - строк, которые полностью идентичных по всем столбцам. Это может быть связано с техническими ошибками при формировании данных, либо дублированием записей при ручном вводе. Наличие дубликатов может исказить результаты анализа, поэтому было принято решение их удалить.

**Удаление дубликатов**

In [40]:
# Удаляем дубликаты
df = df.drop_duplicates(ignore_index=True)
# Выведем количество строк до обработки пропусков и удаления дубликатов (п.2.2)
print(f"Количество строк до обработки пропусков и удаления дубликатов - {total_row_cnt}")
# Выведем полученное количество строк
print(f"Количество строк после обработки - {len(df)}")

Количество строк до обработки пропусков и удаления дубликатов - 16956
Количество строк после обработки - 16444


- Посчитаем количество удалённых строк в абсолютном и относительном значениях.

In [30]:
# Найдем количество удаленных строк
removed_rows = total_row_cnt - len(df)
# Найдем долю удаленных строк в процентах
removed_percent = round((removed_rows/total_row_cnt)*100, 2)
# Выведем полученные значения
print(f"Количество удаленных строк - {removed_rows}, что составляет {removed_percent}% от исходного числа.")

Количество удаленных строк - 512, что составляет 3.02% от исходного числа.


Выведем информацию о датафрейме для подведения промежуточных итогов

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16444 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16444 non-null  object 
 1   platform         16444 non-null  object 
 2   year_of_release  16444 non-null  int16  
 3   genre            16444 non-null  object 
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     16444 non-null  float64
 9   user_score       16444 non-null  float64
 10  rating           9768 non-null   object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.4+ MB


# Промежуточный вывод после проведения предобработки данных

В ходе предобработки данных:

- названия столбцов были приведены к единому стилю;

- изменен тип данных столбцов eu_sales, jp_sales, user_score - float64 и Year_of_Release преобразован в int32;

- обработаны пропуски в столбцах name, genre, eu_sales, jp_sales, year_of_release;

- пропуски в столбцах critic_score и user_score заменены на значение -1, чтобы явно обозначить отсутствие оценки, в столбце rating пропуски оставлены без изменений, так как он не используется в анализе;

- значения в столбцах genre и platform приведены к единому регистру для устранения неявных дубликатов;

- было обнаружено и удалено 235 явных дубликатов;

Исходный датафрейм содержал 11 столбцов и 16596 строк.
Общее количество удаленных строк - 512, что составляет 3.02% от исходного числа. После всех преобразований датафрейм содержит 16444 строк и 11 столбцов. Данные очищены от дубликатов и критических пропусков, приведены к единым форматам и готовы к дальнейшему анализу.

---
<a id="filtration"></a>
## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отбираем данные по этому показателю. Сохраним новый срез данных в отдельном датафрейме, например `df_actual`.

In [32]:
# Отбираем данные за период с 2000 по 2013 год (включительно)
df_actual = df[df['year_of_release'].between(2000, 2013)].copy()
# Выведем на экран размер полученного датафрейма
rows, cols = df_actual.shape
print(f"Количество строк: {rows}")
print(f"Количество столбцов: {cols}")

Количество строк: 12781
Количество столбцов: 11


После фильтрации бы получен срез данных за период с 2000 по 2013 год, состоящий из 12980 строк и 11 столбцов.

---
<a id="categorization"></a>
## 4. Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [33]:
# Функция для категоризации оценок пользователей
def user_score_category(score):
    if score >= 8 and score <=10:
        return 'высокая оценка'
    elif score >= 3:
        return 'средняя оценка'
    elif score >=0:
        return 'низкая оценка'
    else:
        return 'без оценки'

In [34]:
# Применение функции для категоризации, добавление нового столбца
df_actual['user_score_category'] = df_actual['user_score'].apply(user_score_category)

- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [35]:
# Функция для категоризации оценок пользователей
def critic_score_category(score):
    if score >= 80 and score <=100:
        return 'высокая оценка'
    elif score >= 30:
        return 'средняя оценка'
    elif score >=0:
        return 'низкая оценка'
    else:
        return 'без оценки'

In [36]:
# Применение функции для категоризации, добавление нового столбца
df_actual['critic_score_category'] = df_actual['critic_score'].apply(critic_score_category)

- Сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [37]:
# Общее количество игр в срезе
total_games = len(df_actual)

# Группируем данные и считаем количество игр в каждой категории
user_category_cnts = df_actual.groupby('user_score_category')['name'].count()

# Создаем датафрейм с полученными значениями
user_score_df = pd.DataFrame({
    'Количество игр': user_category_cnts,
    'Доля, %': round(user_category_cnts / total_games * 100, 1)
})
# Вывод
print(user_score_df)

                     Количество игр  Доля, %
user_score_category                         
без оценки                     6298     49.3
высокая оценка                 2286     17.9
низкая оценка                   116      0.9
средняя оценка                 4081     31.9


In [38]:
# Группируем данные и считаем количество игр в каждой категории и долю в процентах
critic_category_cnts = df_actual.groupby('critic_score_category')['name'].count()

# Создаем датафрейм с полученными значениями
critic_score_df = pd.DataFrame({
    'Количество игр': critic_category_cnts,
    'Доля, %': round(critic_category_cnts / total_games * 100, 1)
})
# Вывод
print(critic_score_df)

                       Количество игр  Доля, %
critic_score_category                         
без оценки                       5612     43.9
высокая оценка                   1692     13.2
низкая оценка                      55      0.4
средняя оценка                   5422     42.4


После категоризации данных и расчета количества игр в каждой категории оценок, мы получили, что основная масса игр либо не имеет оценок (пользователи - 49.4%, критики - 44%), либо находится в категории "средняя оценка" (пользователи - 32%, критики - 42.4%). Категория игр с высокой оценкой пользователей составляет 17.8%, критиков - 13.2%. Наименьшую долю составляют игры с низкой оценкой (пользователи - 0.4%, критики - 0.9%).

- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [39]:
# Считаем количество игр по платформам и сортируем по убиванию
platform_counts = df_actual.groupby('platform')['name'].count().sort_values(ascending=False)
# Выводим результат
print("Топ-7 платформ по количеству игр за период с 2000 по 2013 год:")
print(platform_counts.head(7))

Топ-7 платформ по количеству игр за период с 2000 по 2013 год:
platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: name, dtype: int64


За период с 2000 по 2013 год наибольшее количество игр было выпущено на платформе PS2 (2154 игры). Топ-7 платформ по количеству игр: PS2, DS, WII, PSP, X360, PS3, GBA.

---
<a id="conclusion"></a>
## 5. Итоговый вывод


Были загружены данные /datasets/new_games.csv. Они содержат 16 956 строк и 11 столбцов с информацией о продажах игр, платформах, жанрах и оценках за период с 1980 по 2016 год. Выявлены особенности типов данных и структуры. 

При первичном знакомстве с данными и их предобработкой получили такие результаты:

- Названия столбцов приведены к единому стилю (snake_case).

- Исправлены типы данных: eu_sales, jp_sales, user_score приведены к float64, year_of_release — к int32.

- Обработаны пропуски: удалены строки с пропусками в ключевых столбцах (name, genre, year_of_release); пропуски в eu_sales и jp_sales заполнены средними значениями по платформе и году; пропуски в critic_score и user_score заменены на -1 как индикатор отсутствия оценки.

- Устранены явные дубликаты (235 строк) и неявные дубликаты в столбце genre (приведение к нижнему регистру).

- Общий объём удалённых строк составил 512 (3.02%), итоговый датафрейм — 16 444 строк.

Для анализа были отобраны данные за период с 2000 по 2013 год включительно, сформирован срез df_actual размером 12 980 строк и 11 столбцов.

Игры разделены на категории по оценкам пользователей и критиков. Выявлено, что большинство игр либо не имеют оценок, либо относятся к категории «средняя оценка».

Определён топ-7 платформ по количеству выпущенных игр: лидирует PS2 (2154 игры), за ней следуют DS, Wii, PSP, X360, PS3, GBA.